
# Production Support Assistant — Multi-Agent LangGraph

A supervisor-style multi-agent system for answering end-user application support
questions.

**Architecture**

```
                 ┌──────────────┐
        ┌───────▶│  Supervisor  │◀───────┐
        │        └──────┬───────┘        │
        │               │ routes to      │
        │      ┌────────┼────────┐       │
        │      ▼         ▼        ▼      │
        │  ┌────────┐┌──────────┐┌────────┐
        └──┤  Jira  ││Confluence││ Splunk │──┘
           │ (MCP)  ││  (MCP)   ││ (API)  │
           └────────┘└──────────┘└────────┘
                        │
                supervisor says FINISH
                        ▼
                 ┌──────────────┐
                 │ Consolidator │──▶ Final answer to user
                 └──────────────┘
```

- **Supervisor**: an LLM router. Looks at the conversation + findings gathered so
  far and decides which specialist to call next, or `FINISH`.
- **Jira agent**: a ReAct agent whose tools come from a **Jira MCP server**
  (via `langchain-mcp-adapters`).
- **Confluence agent**: same pattern, backed by a **Confluence MCP server**.
- **Splunk agent**: a ReAct agent with a custom tool that calls the **Splunk
  REST API** directly and summarizes the returned events/logs.
- **Consolidator**: takes whatever findings were gathered and writes one
  coherent answer back to the user.

Each specialist always reports back to the supervisor, which lets the
supervisor call multiple specialists in sequence (e.g. Jira **then** Splunk)
before finishing — this is the classic LangGraph "supervisor" pattern.

> This notebook uses **OpenAI** models (`langchain-openai`). Swap the LLM
> factory in the Config cell if you want a different provider.


## 1. Install dependencies

In [ ]:
%pip install -qU langgraph langchain langchain-openai langchain-mcp-adapters langgraph-checkpoint requests pydantic

## 2. Configuration

Fill in real endpoints/credentials via environment variables (recommended) or edit the defaults below directly.

In [ ]:
import os

# --- LLM -------------------------------------------------------------------
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
SUPERVISOR_MODEL = os.environ.get("SUPERVISOR_MODEL", "gpt-4o")
SPECIALIST_MODEL = os.environ.get("SPECIALIST_MODEL", "gpt-4o-mini")

# --- Jira MCP server ---------------------------------------------------------
# Point this at your Jira MCP server (Atlassian's official remote MCP server,
# or a self-hosted one). streamable_http is the common transport for remote
# MCP servers; use "stdio" instead if you run a local server process.
JIRA_MCP_URL = os.environ.get("JIRA_MCP_URL", "https://your-jira-mcp-server.example.com/mcp")
JIRA_MCP_TOKEN = os.environ.get("JIRA_MCP_TOKEN", "")

# --- Confluence MCP server ---------------------------------------------------
CONFLUENCE_MCP_URL = os.environ.get("CONFLUENCE_MCP_URL", "https://your-confluence-mcp-server.example.com/mcp")
CONFLUENCE_MCP_TOKEN = os.environ.get("CONFLUENCE_MCP_TOKEN", "")

# --- Splunk REST API ----------------------------------------------------------
SPLUNK_BASE_URL = os.environ.get("SPLUNK_BASE_URL", "https://your-splunk-host:8089")
SPLUNK_TOKEN = os.environ.get("SPLUNK_TOKEN", "")  # HEC or bearer/auth token
SPLUNK_VERIFY_SSL = os.environ.get("SPLUNK_VERIFY_SSL", "true").lower() == "true"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 3. Shared graph state

In [ ]:
from typing import Annotated, Literal, Optional
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


class SupportState(TypedDict):
    # Full running conversation (user turns + specialist/consolidator replies)
    messages: Annotated[list, add_messages]

    # Supervisor's routing decision for the current loop iteration
    next: str

    # Accumulated findings per specialist (used by the consolidator)
    jira_findings: Optional[str]
    confluence_findings: Optional[str]
    splunk_findings: Optional[str]

    # Safety valve so a flaky supervisor can't loop forever
    loop_count: int


MAX_SUPERVISOR_LOOPS = 6

## 4. Connect to the Jira & Confluence MCP servers

`langchain-mcp-adapters` fetches each server's tools as native LangChain tools, which we then hand to a ReAct agent per specialist.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "jira": {
            "transport": "streamable_http",
            "url": JIRA_MCP_URL,
            "headers": {"Authorization": f"Bearer {JIRA_MCP_TOKEN}"},
        },
        "confluence": {
            "transport": "streamable_http",
            "url": CONFLUENCE_MCP_URL,
            "headers": {"Authorization": f"Bearer {CONFLUENCE_MCP_TOKEN}"},
        },
    }
)

# Notebook cells can `await` at the top level.
jira_tools = await mcp_client.get_tools(server_name="jira")
confluence_tools = await mcp_client.get_tools(server_name="confluence")

print(f"Jira tools loaded: {[t.name for t in jira_tools]}")
print(f"Confluence tools loaded: {[t.name for t in confluence_tools]}")

## 5. Splunk specialist (direct API call, no MCP)

In [ ]:
import requests
from langchain_core.tools import tool


@tool
def splunk_search(query: str, earliest_time: str = "-24h", latest_time: str = "now", max_results: int = 50) -> str:
    '''Run a Splunk search query and return matching raw events as JSON.

    Args:
        query: A Splunk search string, e.g. 'search index=app_logs "OrderService" ERROR'.
               Must start with 'search' (or another valid Splunk search command).
        earliest_time: Splunk time modifier for the start of the search window.
        latest_time: Splunk time modifier for the end of the search window.
        max_results: Maximum number of events to return.
    '''
    session_url = f"{SPLUNK_BASE_URL}/services/search/jobs"
    headers = {"Authorization": f"Bearer {SPLUNK_TOKEN}"}

    # 1. Kick off a one-shot search job (oneshot mode returns results directly).
    resp = requests.post(
        session_url,
        headers=headers,
        data={
            "search": query,
            "earliest_time": earliest_time,
            "latest_time": latest_time,
            "exec_mode": "oneshot",
            "output_mode": "json",
            "count": max_results,
        },
        verify=SPLUNK_VERIFY_SSL,
        timeout=30,
    )
    resp.raise_for_status()
    payload = resp.json()

    results = payload.get("results", [])
    if not results:
        return "No matching events found for that query/time range."

    # Trim to keep the tool result compact for the LLM.
    trimmed = results[:max_results]
    return str(trimmed)


splunk_tools = [splunk_search]

## 6. Build the specialist ReAct agents

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

specialist_llm = ChatOpenAI(model=SPECIALIST_MODEL, temperature=0)

jira_agent = create_react_agent(
    specialist_llm,
    tools=jira_tools,
    prompt=(
        "You are a Jira specialist for internal application support. "
        "Use the available tools to look up tickets, statuses, comments, "
        "and history relevant to the user's question. Be precise about ticket "
        "IDs, statuses, and owners. Only report what you actually found via tools."
    ),
)

confluence_agent = create_react_agent(
    specialist_llm,
    tools=confluence_tools,
    prompt=(
        "You are a Confluence specialist for internal application support. "
        "Use the available tools to find relevant documentation, runbooks, "
        "and knowledge-base pages. Summarize what the docs say and cite page "
        "titles. Only report what you actually found via tools."
    ),
)

splunk_agent = create_react_agent(
    specialist_llm,
    tools=splunk_tools,
    prompt=(
        "You are a Splunk/observability specialist for internal application support. "
        "Write a valid Splunk search for the user's question, run it with the "
        "splunk_search tool, and then summarize the findings in plain language: "
        "error patterns, counts, affected services/hosts, and timing. "
        "Only report what the search results actually show."
    ),
)

## 7. Supervisor (router)

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

MEMBERS = ["Jira", "Confluence", "Splunk"]
ROUTE_OPTIONS = MEMBERS + ["FINISH"]

supervisor_llm = ChatOpenAI(model=SUPERVISOR_MODEL, temperature=0)


class RouteDecision(BaseModel):
    next: Literal["Jira", "Confluence", "Splunk", "FINISH"] = Field(
        description="Which specialist to call next, or FINISH if enough information has been gathered."
    )
    reasoning: str = Field(description="One short sentence explaining the choice.")


SUPERVISOR_SYSTEM_PROMPT = (
    "You are the routing supervisor for a production support assistant. "
    "Given the conversation and the findings gathered so far, decide which "
    "specialist should act next:\n"
    "- Jira: ticket status, bugs, incidents, assignees, ticket history.\n"
    "- Confluence: documentation, runbooks, how-to/architecture knowledge.\n"
    "- Splunk: logs, errors, metrics, real-time or historical system behavior.\n"
    "Call FINISH once you have enough findings to fully answer the user, or if "
    "the question doesn't need any specialist (e.g. small talk / clarification).\n"
    "Only route to a specialist that hasn't already produced a useful answer, "
    "unless a follow-up look is genuinely needed."
)


def supervisor_node(state: SupportState) -> dict:
    findings_summary = (
        f"Jira findings so far: {state.get('jira_findings') or 'none'}\n"
        f"Confluence findings so far: {state.get('confluence_findings') or 'none'}\n"
        f"Splunk findings so far: {state.get('splunk_findings') or 'none'}"
    )

    loop_count = state.get("loop_count", 0)
    if loop_count >= MAX_SUPERVISOR_LOOPS:
        return {"next": "FINISH", "loop_count": loop_count + 1}

    messages = [
        SystemMessage(content=SUPERVISOR_SYSTEM_PROMPT),
        *state["messages"],
        SystemMessage(content=findings_summary),
    ]

    decision = supervisor_llm.with_structured_output(RouteDecision).invoke(messages)
    return {"next": decision.next, "loop_count": loop_count + 1}

## 8. Specialist node wrappers

Each wrapper invokes its sub-agent with the conversation so far, stores the finding in state, and appends a labeled message so later nodes (and the supervisor) can see what was learned.

In [ ]:
def _run_specialist(agent, state: SupportState) -> str:
    result = agent.invoke({"messages": state["messages"]})
    return result["messages"][-1].content


def jira_node(state: SupportState) -> dict:
    finding = _run_specialist(jira_agent, state)
    return {
        "messages": [AIMessage(content=finding, name="Jira")],
        "jira_findings": finding,
    }


def confluence_node(state: SupportState) -> dict:
    finding = _run_specialist(confluence_agent, state)
    return {
        "messages": [AIMessage(content=finding, name="Confluence")],
        "confluence_findings": finding,
    }


def splunk_node(state: SupportState) -> dict:
    finding = _run_specialist(splunk_agent, state)
    return {
        "messages": [AIMessage(content=finding, name="Splunk")],
        "splunk_findings": finding,
    }

## 9. Consolidator

In [ ]:
consolidator_llm = ChatOpenAI(model=SUPERVISOR_MODEL, temperature=0.2)

CONSOLIDATOR_SYSTEM_PROMPT = (
    "You are writing the final reply to an end user of an internal support "
    "assistant. You have been given findings from up to three specialists "
    "(Jira, Confluence, Splunk) — some may be empty if that specialist wasn't "
    "needed. Combine only the relevant, non-empty findings into one clear, "
    "well-organized answer. Reference ticket IDs, doc titles, or log details "
    "where helpful. If nothing useful was found anywhere, say so honestly and "
    "suggest a next step (e.g. filing a ticket). Do not invent information."
)


def consolidator_node(state: SupportState) -> dict:
    findings_block = (
        f"Jira findings: {state.get('jira_findings') or 'none'}\n\n"
        f"Confluence findings: {state.get('confluence_findings') or 'none'}\n\n"
        f"Splunk findings: {state.get('splunk_findings') or 'none'}"
    )
    messages = [
        SystemMessage(content=CONSOLIDATOR_SYSTEM_PROMPT),
        *state["messages"],
        SystemMessage(content=findings_block),
    ]
    final = consolidator_llm.invoke(messages)
    return {"messages": [AIMessage(content=final.content, name="Consolidator")]}

## 10. Assemble the graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(SupportState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("Jira", jira_node)
builder.add_node("Confluence", confluence_node)
builder.add_node("Splunk", splunk_node)
builder.add_node("consolidator", consolidator_node)

builder.add_edge(START, "supervisor")

# Every specialist reports back to the supervisor so it can decide whether
# more digging is needed or it's time to consolidate.
for member in MEMBERS:
    builder.add_edge(member, "supervisor")

route_map = {member: member for member in MEMBERS}
route_map["FINISH"] = "consolidator"
builder.add_conditional_edges("supervisor", lambda state: state["next"], route_map)

builder.add_edge("consolidator", END)

# MemorySaver keeps per-thread conversation state so users can ask follow-ups.
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

## 11. Visualize the graph

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## 12. Run it

In [ ]:
config = {"configurable": {"thread_id": "user-123-session-1"}}

initial_state = {
    "messages": [HumanMessage(content="Checkout is throwing 500 errors for some users since this morning, and there's an open ticket about it — what's going on and what's the status?")],
    "next": "",
    "jira_findings": None,
    "confluence_findings": None,
    "splunk_findings": None,
    "loop_count": 0,
}

final_state = graph.invoke(initial_state, config=config)
print(final_state["messages"][-1].content)

## 13. Follow-up turn (same thread, uses checkpointed history)

In [ ]:
followup_state = {
    "messages": [HumanMessage(content="Who is currently assigned to that ticket?")],
    "next": "",
    "loop_count": 0,
}

result = graph.invoke(followup_state, config=config)
print(result["messages"][-1].content)


## Notes for production hardening

- **Auth**: swap the bearer-token headers for whatever your Jira/Confluence MCP
  servers and Splunk deployment actually require (OAuth, mTLS, service
  accounts, etc.). Never hardcode secrets — use env vars or a secrets manager.
- **Streaming**: use `graph.astream(..., stream_mode="messages")` to stream
  tokens to a UI instead of blocking on `invoke`.
- **Guardrails**: `loop_count` caps supervisor re-routing; you may also want a
  timeout per specialist call and a fallback message if a specialist errors
  out (wrap `_run_specialist` in try/except and store an error string as the
  finding instead of raising).
- **Observability**: add `langsmith` tracing (`LANGSMITH_TRACING=true`) to see
  the full routing + tool-call trace per request — very useful for tuning the
  supervisor's prompt.
- **Parallel specialists**: if you'd rather call Jira + Confluence + Splunk
  concurrently instead of one-at-a-time via the supervisor loop, you can use
  `Send` in a conditional edge to fan out to all three at once and let
  LangGraph's state reducers merge results, then always route to
  `consolidator`. The current sequential/supervisor design is simpler to
  reason about and cheaper (skips specialists that aren't relevant).
- **MCP tool scoping**: give the Jira/Confluence MCP server credentials
  read-only, least-privilege access — a support assistant should not be able
  to close tickets or edit pages unless you explicitly want that.
